<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Application-Deployment/blob/main/20261106/ch13_2_MovieLens_1M_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from numpy.random import randn
import numpy as np
np.random.seed(123)
import os
import matplotlib.pyplot as plt
import pandas as pd
plt.rc("figure", figsize=(10, 6))
np.set_printoptions(precision=4)
pd.options.display.max_columns = 20
pd.options.display.max_rows = 20
pd.options.display.max_colwidth = 80

MovieLens 1M 데이터셋(사용자, 영화 정보, 영화 평점 데이터)을 활용하여 성별에 따른 영화 선호도(평점) 차이와 영화별 평점의 호불호(분산/표준편차)를 분석하고, 분석 목적에 맞게 데이터를 가공(원-핫 인코딩 등)하기 위한 탐색적 데이터 분석(EDA) 코드

## **13.2 MovieLens 1M Dataset**

In [7]:
# raw.githubusercontent.com 주소를 사용하여 실제 데이터를 직접 가져옵니다.
raw_path = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/movielens"

unames = ["user_id", "gender", "age", "occupation", "zip"]
users = pd.read_table(f'{raw_path}/users.dat', sep="::",
                      header=None, names=unames, engine="python")
users

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [10]:
raw_path = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/movielens"

rnames = ["user_id", "movie_id", "rating", "timestamp"]
ratings = pd.read_table(f'{raw_path}/ratings.dat', sep="::",
                        header=None, names=rnames, engine="python")

# 전체 데이터 중 10%만 무작위로 추출하여 ratings 변수에 재할당
# ratings = ratings .sample(frac=0.05, random_state=42)
ratings

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [12]:
mnames = ["movie_id", "title", "genres"]
movies = pd.read_table(f'{raw_path}/movies.dat', sep="::",
                       header=None, names=mnames, engine="python")
# 전체 데이터 중 10%만 무작위로 추출하여 ratings 변수에 재할당
# movies = movies.sample(frac=0.05, random_state=42)
movies

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [82]:
# @title 모든 cell의 명령문 출력을 한꺼번에 보기
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [15]:
users.head(5)
ratings.head(5)
movies.head(5)

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [16]:
# @title **3개 파일을 정리하여 데이터를 파악하는 코드**

def summarize_dataframe(df, df_name):
    print(f"\n--- Summary for DataFrame: {df_name} ---")
    summary = pd.DataFrame(index=df.columns)
    summary["Data Type"] = df.dtypes
    summary["Unique Values"] = df.nunique()
    summary["Missing Values (%)"] = df.isnull().sum() * 100 / len(df)

    top_values_list = []
    for col in df.columns:
        if df[col].nunique() > 0:
            top_5_values = df[col].value_counts().nlargest(5).to_dict()
            top_values_list.append(str(top_5_values))
        else:
            top_values_list.append("N/A")

    summary["Top 5 Values"] = top_values_list

    # Convert float columns to two decimal places where appropriate
    for col in ["Missing Values (%)"]:
        if col in summary.columns:
            summary[col] = summary[col].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "N/A")

    display(summary)

# Apply the function to the specified DataFrames
summarize_dataframe(users, "users")
summarize_dataframe(ratings, "ratings")
summarize_dataframe(movies, "movies")


--- Summary for DataFrame: users ---


,Data Type,Unique Values,Missing Values (%),Top 5 Values
user_id,int64,6040,0.00,"{6040: 1, 1: 1, 2: 1, 3: 1, 4: 1}"
gender,object,2,0.00,"{'M': 4331, 'F': 1709}"
age,int64,7,0.00,"{25: 2096, 35: 1193, 18: 1103, 45: 550, 50: 496}"
occupation,int64,21,0.00,"{4: 759, 0: 711, 7: 679, 1: 528, 17: 502}"
zip,object,3439,0.00,"{'48104': 19, '22903': 18, '55104': 17, '94110': 17, '10025': 16}"



--- Summary for DataFrame: ratings ---


,Data Type,Unique Values,Missing Values (%),Top 5 Values
user_id,int64,6040,0.00,"{4169: 2314, 1680: 1850, 4277: 1743, 1941: 1595, 1181: 1521}"
movie_id,int64,3706,0.00,"{2858: 3428, 260: 2991, 1196: 2990, 1210: 2883, 480: 2672}"
rating,int64,5,0.00,"{4: 348971, 3: 261197, 5: 226310, 2: 107557, 1: 56174}"
timestamp,int64,458455,0.00,"{975528402: 30, 975440712: 28, 975527781: 28, 975528243: 27, 1025585635: 27}"



--- Summary for DataFrame: movies ---


,Data Type,Unique Values,Missing Values (%),Top 5 Values
movie_id,int64,3883,0.00,"{3952: 1, 1: 1, 2: 1, 3: 1, 4: 1}"
title,object,3883,0.00,"{'Contender, The (2000)': 1, 'Toy Story (1995)': 1, 'Jumanji (1995)': 1, 'Gr..."
genres,object,301,0.00,"{'Drama': 843, 'Comedy': 521, 'Horror': 178, 'Comedy|Drama': 162, 'Comedy|Ro..."


In [17]:
data = pd.merge(pd.merge(ratings, users), movies)
data
data.iloc[0]

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama


,0
user_id,1
movie_id,1193
rating,5
timestamp,978300760
gender,F
age,1
occupation,10
zip,48067
title,One Flew Over the Cuckoo's Nest (1975)
genres,Drama


In [18]:
summarize_dataframe(data, 'data')


--- Summary for DataFrame: data ---


,Data Type,Unique Values,Missing Values (%),Top 5 Values
user_id,int64,6040,0.00,"{4169: 2314, 1680: 1850, 4277: 1743, 1941: 1595, 1181: 1521}"
movie_id,int64,3706,0.00,"{2858: 3428, 260: 2991, 1196: 2990, 1210: 2883, 480: 2672}"
rating,int64,5,0.00,"{4: 348971, 3: 261197, 5: 226310, 2: 107557, 1: 56174}"
timestamp,int64,458455,0.00,"{975528402: 30, 975440712: 28, 975527781: 28, 975528243: 27, 1025585635: 27}"
gender,object,2,0.00,"{'M': 753769, 'F': 246440}"
age,int64,7,0.00,"{25: 395556, 35: 199003, 18: 183536, 45: 83633, 50: 72490}"
occupation,int64,21,0.00,"{4: 131032, 0: 130499, 7: 105425, 1: 85351, 17: 72816}"
zip,object,3439,0.00,"{'94110': 3802, '60640': 3430, '98103': 3204, '95616': 3079, '02138': 3019}"
title,object,3706,0.00,"{'American Beauty (1999)': 3428, 'Star Wars: Episode IV - A New Hope (1977)'..."
genres,object,301,0.00,"{'Comedy': 116883, 'Drama': 111423, 'Comedy|Romance': 42712, 'Comedy|Drama':..."


In [83]:
data

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1091,1,956716541,M,25,6,11106,Weekend at Bernie's (1989),Comedy
1000205,6040,1094,5,956704887,M,25,6,11106,"Crying Game, The (1992)",Drama|Romance|War
1000206,6040,562,5,956704746,M,25,6,11106,Welcome to the Dollhouse (1995),Comedy|Drama
1000207,6040,1096,4,956715648,M,25,6,11106,Sophie's Choice (1982),Drama


In [84]:
# @title 영화 제목별로 몇 명의 시청자가 관람하였는지?
ratings_by_title = data.groupby("title").size()
ratings_by_title

,0
title,
"$1,000,000 Duck (1971)",37
'Night Mother (1986),70
'Til There Was You (1997),52
"'burbs, The (1989)",303
...And Justice for All (1979),199
...,...
"Zed & Two Noughts, A (1985)",29
Zero Effect (1998),301
Zero Kelvin (Kjærlighetens kjøtere) (1995),2


In [24]:
data[data['title'] == '$1,000,000 Duck (1971)']

,user_id,movie_id,rating,timestamp,gender,age,occupation,zip,title,genres
32396,216,2031,2,976867230,M,45,13,52761,"$1,000,000 Duck (1971)",Children's|Comedy
73210,494,2031,5,976215651,F,35,0,17870,"$1,000,000 Duck (1971)",Children's|Comedy
109519,714,2031,4,975782711,M,18,4,76013,"$1,000,000 Duck (1971)",Children's|Comedy
134719,869,2031,1,999376619,M,18,20,92026,"$1,000,000 Duck (1971)",Children's|Comedy
162426,1034,2031,3,975093319,F,35,1,82601,"$1,000,000 Duck (1971)",Children's|Comedy
...,...,...,...,...,...,...,...,...,...,...
861075,5179,2031,1,961807264,M,25,12,94110,"$1,000,000 Duck (1971)",Children's|Comedy
862224,5185,2031,3,963068261,F,35,4,44485,"$1,000,000 Duck (1971)",Children's|Comedy
935538,5643,2031,2,984200692,F,35,1,84108,"$1,000,000 Duck (1971)",Children's|Comedy
948666,5728,2031,3,958475031,F,35,20,09824,"$1,000,000 Duck (1971)",Children's|Comedy


In [86]:
data.groupby(["user_id", "title"]).size()

user_id  title                                           
1        Airplane! (1980)                                    1
         Aladdin (1992)                                      1
         Antz (1998)                                         1
         Apollo 13 (1995)                                    1
         Awakenings (1990)                                   1
                                                            ..
6040     Wizard of Oz, The (1939)                            1
         Woman in the Dunes (Suna no onna) (1964)            1
         Women on the Verge of a Nervous Breakdown (1988)    1
         Year of Living Dangerously (1982)                   1
         Young Frankenstein (1974)                           1
Length: 1000209, dtype: int64

In [87]:
data.groupby(["user_id", "title"]).size().reset_index()

,user_id,title,0
0,1,Airplane! (1980),1
1,1,Aladdin (1992),1
2,1,Antz (1998),1
3,1,Apollo 13 (1995),1
4,1,Awakenings (1990),1
...,...,...,...
1000204,6040,"Wizard of Oz, The (1939)",1
1000205,6040,Woman in the Dunes (Suna no onna) (1964),1
1000206,6040,Women on the Verge of a Nervous Breakdown (1988),1
1000207,6040,Year of Living Dangerously (1982),1


In [90]:
# @title 특정 사용자가 동일한 영화에 대해 중복해서 평점을 남겼거나 기록이 중복 존재하는지 검사
tf = data.groupby(["user_id", "title"]).size().reset_index()
tf[tf.duplicated()]

,user_id,title,0


In [91]:
# @title 영화 평점이 250 이상인 영화제목
active_titles = ratings_by_title.index[ratings_by_title >= 250]
active_titles

Index([''burbs, The (1989)', '10 Things I Hate About You (1999)',
       '101 Dalmatians (1961)', '101 Dalmatians (1996)', '12 Angry Men (1957)',
       '13th Warrior, The (1999)', '2 Days in the Valley (1996)',
       '20,000 Leagues Under the Sea (1954)', '2001: A Space Odyssey (1968)',
       '2010 (1984)',
       ...
       'X-Men (2000)', 'Year of Living Dangerously (1982)',
       'Yellow Submarine (1968)', 'You've Got Mail (1998)',
       'Young Frankenstein (1974)', 'Young Guns (1988)',
       'Young Guns II (1990)', 'Young Sherlock Holmes (1985)',
       'Zero Effect (1998)', 'eXistenZ (1999)'],
      dtype='object', name='title', length=1216)

In [92]:
# @title 성별로 영화평점 평균점수
mean_ratings = data.pivot_table("rating", index="title",
                                columns="gender", aggfunc="mean")
mean_ratings

gender,F,M
title,,
"$1,000,000 Duck (1971)",3.375000,2.761905
'Night Mother (1986),3.388889,3.352941
'Til There Was You (1997),2.675676,2.733333
"'burbs, The (1989)",2.793478,2.962085
...And Justice for All (1979),3.828571,3.689024
...,...,...
"Zed & Two Noughts, A (1985)",3.500000,3.380952
Zero Effect (1998),3.864407,3.723140
Zero Kelvin (Kjærlighetens kjøtere) (1995),NaN,3.500000


In [30]:
mean_ratings

gender,F,M
title,,
"$1,000,000 Duck (1971)",3.375000,2.761905
'Night Mother (1986),3.388889,3.352941
'Til There Was You (1997),2.675676,2.733333
"'burbs, The (1989)",2.793478,2.962085
...And Justice for All (1979),3.828571,3.689024
...,...,...
"Zed & Two Noughts, A (1985)",3.500000,3.380952
Zero Effect (1998),3.864407,3.723140
Zero Kelvin (Kjærlighetens kjøtere) (1995),NaN,3.500000


In [32]:
mean_ratings.loc['eXistenZ (1999)']

,eXistenZ (1999)
gender,
F,3.098592
M,3.289086


In [34]:
active_titles

Index([''burbs, The (1989)', '10 Things I Hate About You (1999)',
       '101 Dalmatians (1961)', '101 Dalmatians (1996)', '12 Angry Men (1957)',
       '13th Warrior, The (1999)', '2 Days in the Valley (1996)',
       '20,000 Leagues Under the Sea (1954)', '2001: A Space Odyssey (1968)',
       '2010 (1984)',
       ...
       'X-Men (2000)', 'Year of Living Dangerously (1982)',
       'Yellow Submarine (1968)', 'You've Got Mail (1998)',
       'Young Frankenstein (1974)', 'Young Guns (1988)',
       'Young Guns II (1990)', 'Young Sherlock Holmes (1985)',
       'Zero Effect (1998)', 'eXistenZ (1999)'],
      dtype='object', name='title', length=1216)

In [35]:
# @title 영화평점이 250점 이상인 영화에 대한 성별에 따른 평균 평점
mean_ratings = mean_ratings.loc[active_titles]
mean_ratings

gender,F,M
title,,
"'burbs, The (1989)",2.793478,2.962085
10 Things I Hate About You (1999),3.646552,3.311966
101 Dalmatians (1961),3.791444,3.500000
101 Dalmatians (1996),3.240000,2.911215
12 Angry Men (1957),4.184397,4.328421
...,...,...
Young Guns (1988),3.371795,3.425620
Young Guns II (1990),2.934783,2.904025
Young Sherlock Holmes (1985),3.514706,3.363344


In [36]:
# @title 특정 영화 제목 변경
mean_ratings = mean_ratings.rename(index={"Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)":
                           "Seven Samurai (Shichinin no samurai) (1954)"})

mean_ratings

gender,F,M
title,,
"'burbs, The (1989)",2.793478,2.962085
10 Things I Hate About You (1999),3.646552,3.311966
101 Dalmatians (1961),3.791444,3.500000
101 Dalmatians (1996),3.240000,2.911215
12 Angry Men (1957),4.184397,4.328421
...,...,...
Young Guns (1988),3.371795,3.425620
Young Guns II (1990),2.934783,2.904025
Young Sherlock Holmes (1985),3.514706,3.363344


In [37]:
top_female_ratings = mean_ratings.sort_values("F", ascending=False)
top_female_ratings.head()

gender,F,M
title,,
"Close Shave, A (1995)",4.644444,4.473795
"Wrong Trousers, The (1993)",4.588235,4.478261
Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),4.572650,4.464589
Wallace & Gromit: The Best of Aardman Animation (1996),4.563107,4.385075
Schindler's List (1993),4.562602,4.491415


In [38]:
mean_ratings["diff"] = mean_ratings["M"] - mean_ratings["F"]

In [39]:
sorted_by_diff = mean_ratings.sort_values("diff")
sorted_by_diff.head()

gender,F,M,diff
title,,,
Dirty Dancing (1987),3.790378,2.959596,-0.830782
Jumpin' Jack Flash (1986),3.254717,2.578358,-0.676359
Grease (1978),3.975265,3.367041,-0.608224
Little Women (1994),3.870588,3.321739,-0.548849
Steel Magnolias (1989),3.901734,3.365957,-0.535777


In [40]:
sorted_by_diff[::-1].head()

gender,F,M,diff
title,,,
"Good, The Bad and The Ugly, The (1966)",3.494949,4.221300,0.726351
"Kentucky Fried Movie, The (1977)",2.878788,3.555147,0.676359
Dumb & Dumber (1994),2.697987,3.336595,0.638608
"Longest Day, The (1962)",3.411765,4.031447,0.619682
"Cable Guy, The (1996)",2.250000,2.863787,0.613787


In [41]:
rating_std_by_title = data.groupby("title")["rating"].std()
rating_std_by_title = rating_std_by_title.loc[active_titles]
rating_std_by_title.head()

,rating
title,
"'burbs, The (1989)",1.107760
10 Things I Hate About You (1999),0.989815
101 Dalmatians (1961),0.982103
101 Dalmatians (1996),1.098717
12 Angry Men (1957),0.812731


In [42]:
rating_std_by_title.sort_values(ascending=False)[:10]

,rating
title,
Dumb & Dumber (1994),1.321333
"Blair Witch Project, The (1999)",1.316368
Natural Born Killers (1994),1.307198
Tank Girl (1995),1.277695
"Rocky Horror Picture Show, The (1975)",1.260177
Eyes Wide Shut (1999),1.259624
Evita (1996),1.253631
Billy Madison (1995),1.249970
Fear and Loathing in Las Vegas (1998),1.246408


영화 장르(genres)를 다중 장르로 분리(explode)하고 원-핫 인코딩(get_dummies)을 적용하여, 머신러닝 모델 입력이나 장르별 추가 분석에 사용할 수 있도록 데이터를 정제

In [43]:
movies["genres"].head()
movies["genres"].head().str.split("|")
movies["genre"] = movies.pop("genres").str.split("|")
movies.head()

,genres
0,Animation|Children's|Comedy
1,Adventure|Children's|Fantasy
2,Comedy|Romance
3,Comedy|Drama
4,Comedy


,genres
0,"[Animation, Children's, Comedy]"
1,"[Adventure, Children's, Fantasy]"
2,"[Comedy, Romance]"
3,"[Comedy, Drama]"
4,[Comedy]


,movie_id,title,genre
0,1,Toy Story (1995),"[Animation, Children's, Comedy]"
1,2,Jumanji (1995),"[Adventure, Children's, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama]"
4,5,Father of the Bride Part II (1995),[Comedy]


In [44]:
movies_exploded = movies.explode("genre")
movies_exploded[:10]

,movie_id,title,genre
0,1,Toy Story (1995),Animation
0,1,Toy Story (1995),Children's
0,1,Toy Story (1995),Comedy
1,2,Jumanji (1995),Adventure
1,2,Jumanji (1995),Children's
1,2,Jumanji (1995),Fantasy
2,3,Grumpier Old Men (1995),Comedy
2,3,Grumpier Old Men (1995),Romance
3,4,Waiting to Exhale (1995),Comedy
3,4,Waiting to Exhale (1995),Drama


In [61]:
movies_exploded['genre'].unique()

array(['Animation', "Children's", 'Comedy', 'Adventure', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Sci-Fi', 'Documentary', 'War', 'Musical', 'Mystery', 'Film-Noir',
       'Western'], dtype=object)

In [62]:
movies_exploded['genre'].nunique()

18

In [47]:
# @title **모든 장르를 컬럼으로하고 해당이 되면 0 아니면 1로 값을 입력하는 코드**
genres_one_hot = pd.get_dummies(movies_exploded['genre'], prefix='genre')
genres_one_hot

,genre_Action,genre_Adventure,genre_Animation,genre_Children's,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3879,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
3880,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
3881,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
3882,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False


In [64]:
movies_exploded[['movie_id', 'title']].head(10)

,movie_id,title
0,1,Toy Story (1995)
0,1,Toy Story (1995)
0,1,Toy Story (1995)
1,2,Jumanji (1995)
1,2,Jumanji (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
3,4,Waiting to Exhale (1995)


In [49]:
movies_exploded[['movie_id', 'title']].drop_duplicates()

,movie_id,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)
...,...,...
3878,3948,Meet the Parents (2000)
3879,3949,Requiem for a Dream (2000)
3880,3950,Tigerland (2000)
3881,3951,Two Family House (2000)


In [50]:
movies_exploded[['movie_id', 'title']].drop_duplicates().set_index('movie_id')

,title
movie_id,
1,Toy Story (1995)
2,Jumanji (1995)
3,Grumpier Old Men (1995)
4,Waiting to Exhale (1995)
5,Father of the Bride Part II (1995)
...,...
3948,Meet the Parents (2000)
3949,Requiem for a Dream (2000)
3950,Tigerland (2000)


In [51]:
movies_with_genres = movies_exploded[['movie_id', 'title']].drop_duplicates().set_index('movie_id')
movies_with_genres

,title
movie_id,
1,Toy Story (1995)
2,Jumanji (1995)
3,Grumpier Old Men (1995)
4,Waiting to Exhale (1995)
5,Father of the Bride Part II (1995)
...,...
3948,Meet the Parents (2000)
3949,Requiem for a Dream (2000)
3950,Tigerland (2000)


In [59]:
genres_one_hot.head(10)

,genre_Action,genre_Adventure,genre_Animation,genre_Children's,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False


In [65]:
movies_exploded[['movie_id', 'title']].index

Index([   0,    0,    0,    1,    1,    1,    2,    2,    3,    3,
       ...
       3876, 3876, 3876, 3877, 3878, 3879, 3880, 3881, 3882, 3882],
      dtype='int64', length=6408)

In [58]:
genres_one_hot.index

Index([   0,    0,    0,    1,    1,    1,    2,    2,    3,    3,
       ...
       3876, 3876, 3876, 3877, 3878, 3879, 3880, 3881, 3882, 3882],
      dtype='int64', length=6408)

In [60]:
# Merge the one-hot encoded genres with movie_id and title
movies_genres_encoded = pd.merge(movies_exploded[['movie_id', 'title']], genres_one_hot, left_index=True, right_index=True)
movies_genres_encoded.head(10)

,movie_id,title,genre_Action,genre_Adventure,genre_Animation,genre_Children's,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,1,Toy Story (1995),False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,1,Toy Story (1995),False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
0,1,Toy Story (1995),False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,1,Toy Story (1995),False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,1,Toy Story (1995),False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
0,1,Toy Story (1995),False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,1,Toy Story (1995),False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
0,1,Toy Story (1995),False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2,Jumanji (1995),False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [69]:
# Group by movie_id and title, then sum the genre columns to handle movies with multiple genres
final_movies_genres = movies_genres_encoded.groupby(['movie_id', 'title']).sum().reset_index()
final_movies_genres

,movie_id,title,genre_Action,genre_Adventure,genre_Animation,genre_Children's,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),0,0,3,3,3,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,3,0,3,0,0,0,0,3,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,2,0,0,0,0,0,0,0,0,2,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,2,0,0,2,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3878,3948,Meet the Parents (2000),0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3879,3949,Requiem for a Dream (2000),0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3880,3950,Tigerland (2000),0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3881,3951,Two Family House (2000),0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [72]:
(final_movies_genres['genre_Action'] > 0).astype(int)

,genre_Action
0,0
1,0
2,0
3,0
4,0
...,...
3878,0
3879,0
3880,0
3881,0


In [73]:
(final_movies_genres['genre_Animation'] > 0).astype(int)

,genre_Animation
0,1
1,0
2,0
3,0
4,0
...,...
3878,0
3879,0
3880,0
3881,0


In [56]:
# Convert sums greater than 1 to 1 for true binary representation (though get_dummies + sum should result in 0 or 1 if genre is unique per row)
# In case a movie had the same genre listed multiple times, this step ensures it's still 1.
for col in final_movies_genres.columns:
    if col.startswith('genre_'):
        final_movies_genres[col] = (final_movies_genres[col] > 0).astype(int)

final_movies_genres

,movie_id,title,genre_Action,genre_Adventure,genre_Animation,genre_Children's,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3878,3948,Meet the Parents (2000),0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3879,3949,Requiem for a Dream (2000),0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3880,3950,Tigerland (2000),0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3881,3951,Two Family House (2000),0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [77]:
ratings_with_genre = pd.merge(pd.merge(movies_exploded, ratings), users)
ratings_with_genre
ratings_with_genre.iloc[0]

,movie_id,title,genre,user_id,rating,timestamp,gender,age,occupation,zip
0,1,Toy Story (1995),Animation,1,5,978824268,F,1,10,48067
1,1,Toy Story (1995),Animation,6,4,978237008,F,50,9,55117
2,1,Toy Story (1995),Animation,8,4,978233496,M,25,12,11413
3,1,Toy Story (1995),Animation,9,5,978225952,M,25,17,61614
4,1,Toy Story (1995),Animation,10,5,978226474,F,35,1,95370
...,...,...,...,...,...,...,...,...,...,...
2101810,3952,"Contender, The (2000)",Thriller,5812,4,992072099,F,25,7,92120
2101811,3952,"Contender, The (2000)",Thriller,5831,3,986223125,M,25,1,92120
2101812,3952,"Contender, The (2000)",Thriller,5837,4,1011902656,M,25,7,60607
2101813,3952,"Contender, The (2000)",Thriller,5927,1,979852537,M,35,14,10003


,0
movie_id,1
title,Toy Story (1995)
genre,Animation
user_id,1
rating,5
timestamp,978824268
gender,F
age,1
occupation,10
zip,48067


In [78]:
ratings_with_genre.groupby(["genre", "age"])["rating"].mean()

genre    age
Action   1      3.506385
         18     3.447097
         25     3.453358
         35     3.538107
         45     3.528543
                  ...   
Western  25     3.607826
         35     3.679278
         45     3.667135
         50     3.741322
         56     3.792198
Name: rating, Length: 126, dtype: float64

In [79]:
ratings_with_genre.groupby(["genre", "age"])["rating"].mean().unstack()

age,1,18,25,35,45,50,56
genre,,,,,,,
Action,3.506385,3.447097,3.453358,3.538107,3.528543,3.611333,3.610709
Adventure,3.449975,3.408525,3.443163,3.515291,3.528963,3.628163,3.649064
Animation,3.476113,3.624014,3.701228,3.740545,3.734856,3.780020,3.756233
Children's,3.241642,3.294257,3.426873,3.518423,3.527593,3.556555,3.621822
Comedy,3.497491,3.460417,3.490385,3.561984,3.591789,3.646868,3.650949
Crime,3.710170,3.668054,3.680321,3.733736,3.750661,3.810688,3.832549
Documentary,3.730769,3.865865,3.946690,3.953747,3.966521,3.908108,3.961538
Drama,3.794735,3.721930,3.726428,3.782512,3.784356,3.878415,3.933465
Fantasy,3.317647,3.353778,3.452484,3.482301,3.532468,3.581570,3.532700


In [81]:
genre_ratings = (ratings_with_genre.groupby(["genre", "age"])["rating"].mean().unstack("age"))
genre_ratings

age,1,18,25,35,45,50,56
genre,,,,,,,
Action,3.506385,3.447097,3.453358,3.538107,3.528543,3.611333,3.610709
Adventure,3.449975,3.408525,3.443163,3.515291,3.528963,3.628163,3.649064
Animation,3.476113,3.624014,3.701228,3.740545,3.734856,3.780020,3.756233
Children's,3.241642,3.294257,3.426873,3.518423,3.527593,3.556555,3.621822
Comedy,3.497491,3.460417,3.490385,3.561984,3.591789,3.646868,3.650949
Crime,3.710170,3.668054,3.680321,3.733736,3.750661,3.810688,3.832549
Documentary,3.730769,3.865865,3.946690,3.953747,3.966521,3.908108,3.961538
Drama,3.794735,3.721930,3.726428,3.782512,3.784356,3.878415,3.933465
Fantasy,3.317647,3.353778,3.452484,3.482301,3.532468,3.581570,3.532700
